# Merge CSVs
Keep every GovEsp record and append only GenCat rows with no matching municipality and calendar day. A match means **any** GovEsp row at that municipality/day; multiple incidents on the same day are not paired one-to-one.

Unify dates, IDs and column names. Municipality codes become full five-digit codes (province + local code when needed). Use those codes to reconcile name variants such as `Escala, l'` / `L'Escala` for matching, while retaining the original names in the output. Strip case/whitespace differences. Convert decimal-comma coordinates to numbers; preserve missing data and source-specific columns.

The current GenCat file has a header-only Git conflict: read the incoming header in memory, without editing either source CSV.

In [1]:
from pathlib import Path
from io import StringIO
import pandas as pd

folder = Path("Clean data")
if not folder.is_dir():
    folder = Path("VI_Project1/Clean data")


def read_clean(filename):
    lines = (folder / filename).read_text(encoding="utf-8-sig").splitlines()
    if lines[0].startswith("<<<<<<<"):  # Current conflict affects only the header.
        assert lines[2] == "=======" and lines[4].startswith(">>>>>>>")
        lines = [lines[3], *lines[5:]]
    assert not any(line.startswith(("<<<<<<<", "=======", ">>>>>>>")) for line in lines)
    return pd.read_csv(StringIO("\n".join(lines))).filter(regex=r"^(?!Unnamed:)")

gencat, govesp = read_clean("Clean GenCat Data.csv"), read_clean("CleanGovEsp.csv")
column_names = {
    "numeroparte": "id_incendi",
    "idincident": "id_incendi",
    "data": "data_incendi",
    "idprovincia": "codi_provincia",
    "idmunicipio": "codi_municipi",
    "idcausa": "codi_causa",
    "coords_imputades": "coordenades_imputades",
}
for data, source in [(gencat, "GenCat"), (govesp, "GovEsp")]:
    data.rename(columns=column_names, inplace=True)
    data["data_incendi"] = pd.to_datetime(data["data_incendi"], errors="raise").dt.normalize()
    province = pd.to_numeric(data["codi_provincia"], errors="raise").astype("Int64")
    code = pd.to_numeric(data["codi_municipi"], errors="raise").astype("Int64")
    # Local municipality codes need their province prefix.
    full_code = code.where(code.ge(1000), province * 1000 + code)
    data["codi_municipi"] = full_code.astype("string").str.zfill(5)
    data["codi_provincia"] = province.astype("string").str.zfill(2)
    data["id_incendi"] = source + ":" + data["id_incendi"].astype("string")
    data["font"] = source
    for column in ["latitud", "longitud"]:
        decimal_values = data[column].astype("string").str.replace(",", ".", regex=False)
        data[column] = pd.to_numeric(decimal_values, errors="raise")

## Match and check the hypothesis
`unmatched_gencat` records every GenCat exception. Separate common calendar years from years covered only by GenCat; later records are not evidence of disagreement. Also count GovEsp-only records in common years and shared municipality/day groups with different incident counts. Missing municipality names cannot match.

In [2]:
municipality_names = gencat.drop_duplicates("codi_municipi").set_index("codi_municipi")["municipi"]
match_columns = ["_municipi", "data_incendi"]
for data in [gencat, govesp]:
    names = data["codi_municipi"].map(municipality_names).fillna(data["municipi"])
    data["_municipi"] = (
        names.astype("string")
        .str.normalize("NFKC")
        .str.casefold()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

gencat_days = pd.MultiIndex.from_frame(gencat[match_columns])
govesp_days = pd.MultiIndex.from_frame(govesp[match_columns].dropna())
matched = gencat_days.isin(govesp_days)
unmatched_gencat = gencat.loc[~matched].drop(columns="_municipi").copy()
start = max(gencat.data_incendi.dt.year.min(), govesp.data_incendi.dt.year.min())
end = min(gencat.data_incendi.dt.year.max(), govesp.data_incendi.dt.year.max())
common = gencat.data_incendi.dt.year.between(start, end)
gov_common = govesp.loc[govesp.data_incendi.dt.year.between(start, end)]
gov_common_days = pd.MultiIndex.from_frame(gov_common[match_columns])
gencat_known_days = pd.MultiIndex.from_frame(gencat[match_columns].dropna())
gov_only = ~gov_common_days.isin(gencat_known_days)

counts = pd.concat(
    [gencat.groupby(match_columns).size(), gov_common.groupby(match_columns).size()],
    axis=1,
    keys=["GenCat", "GovEsp"],
).dropna()
summary = pd.Series({
    "GenCat input": len(gencat),
    "GovEsp input": len(govesp),
    "GenCat matched": int(matched.sum()),
    "GenCat unmatched (all years)": len(unmatched_gencat),
    f"GenCat unmatched ({start}-{end})": int((~matched & common).sum()),
    "GenCat unmatched outside common years": int((~matched & ~common).sum()),
    f"GovEsp unmatched ({start}-{end})": int(gov_only.sum()),
    "Shared municipality/day groups with different counts": int(counts.GenCat.ne(counts.GovEsp).sum()),
})
print(summary.to_string())
print("\nFirst 10 unmatched GenCat records (all available in unmatched_gencat):")
print(unmatched_gencat[["id_incendi", "municipi", "data_incendi"]].head(10).to_string(index=False))

GenCat input                                             7448
GovEsp input                                            15366
GenCat matched                                           6343
GenCat unmatched (all years)                             1105
GenCat unmatched (2011-2022)                               60
GenCat unmatched outside common years                    1045
GovEsp unmatched (2011-2022)                               54
Shared municipality/day groups with different counts       53

First 10 unmatched GenCat records (all available in unmatched_gencat):
 id_incendi               municipi data_incendi
GenCat:2026    Isona i Conca Dellà   2014-03-12
GenCat:2226                Tivissa   2014-07-01
GenCat:2449 Sant Vicenç de Torelló   2015-03-03
GenCat:2713  Sant Feliu de Guíxols   2015-06-24
GenCat:2923       Vimbodí i Poblet   2016-01-13
GenCat:3070               Alcarràs   2016-04-24
GenCat:3560          Molins de Rei   2017-04-09
GenCat:3583         Granadella, la   2017-04-21


## Consolidate the 18 July 1998 Aguilar de Segarra fire
The GovEsp records `1998080173` (Barcelona) and `1998251111` (Lleida) cover separate provincial portions of a single fire. Sum their burned hectares and retain the Aguilar de Segarra origin record for incident counts and location. The original rows remain available in `CleanGovEsp.csv`. The 19 July Cardona fire is a separate incident.

[MITECO 1996–2005 wildfire report, tables 17 and 19](https://miteco.gob.es/content/dam/miteco/es/biodiversidad/temas/incendios-forestales/decenio_1996_2005_tcm30-132558.pdf)

In [3]:
merged = pd.concat(
    [govesp.drop(columns="_municipi"), unmatched_gencat],
    ignore_index=True,
    sort=False,
)
assert len(merged) == len(govesp) + len(unmatched_gencat)
assert merged.id_incendi.is_unique

origin_id = "GovEsp:1998080173"
lleida_id = "GovEsp:1998251111"
aguilar = merged.loc[merged.id_incendi.isin([origin_id, lleida_id])].set_index("id_incendi")
if (
    len(aguilar) != 2
    or not aguilar.data_incendi.eq(pd.Timestamp("1998-07-18")).all()
    or aguilar[["latitud", "longitud", "codi_causa"]].nunique().ne(1).any()
    or not aguilar.total_hectares.gt(0).all()
):
    raise ValueError("Unexpected source records for the Aguilar de Segarra consolidation.")

combined_hectares = round(aguilar.total_hectares.sum(), 2)
merged.loc[merged.id_incendi.eq(origin_id), "total_hectares"] = combined_hectares
merged = merged.loc[merged.id_incendi.ne(lleida_id)].copy()
print(f"Aguilar de Segarra: combined {origin_id} and {lleida_id} into one incident ({combined_hectares:,.2f} ha).")


Aguilar de Segarra: combined GovEsp:1998080173 and GovEsp:1998251111 into one incident (17,675.18 ha).


## Export
`CleanData.csv` contains GovEsp rows plus unmatched GenCat rows, sorted by date, with the documented Aguilar de Segarra consolidation applied. `font` records provenance; prefixed `id_incendi` values avoid collisions between source IDs. `codi_municipi` identifies the full municipality, and `codi_provincia` the province. GenCat-only cause fields and GovEsp-only comarca fields remain blank.
GovEsp `total_hectares` is retained; it remains blank for GenCat-only records without area data.

`codi_comarca` comes from the Idescat municipality-to-comarca lookup used for the project's 2024 geography. Codes are stored as strings to preserve leading zeroes. Fires without a match are retained with a blank code and counted below.

In [4]:
correspondence = pd.read_csv(
    folder.parent / "dades" / "corrMpisComsAtps.csv",
    sep=";", skiprows=3, encoding="utf-8-sig", dtype="string",
    usecols=["Codi municipi", "Codi comarca"],
)
comarca_lookup = correspondence.set_index(correspondence["Codi municipi"].str[:5])["Codi comarca"]
merged["codi_comarca"] = merged["codi_municipi"].map(comarca_lookup).astype("string")
print(f"Records without a comarca code: {merged['codi_comarca'].isna().sum():,}")

export_columns = [
    "id_incendi", "font", "data_incendi",
    "codi_provincia", "codi_municipi", "municipi", "provincia", "codi_comarca", "comarca",
    "latitud", "longitud", "coordenades_imputades",
    "codi_causa", "causa", "total_hectares",
]
merged = (
    merged.reindex(columns=export_columns)
    .sort_values(["data_incendi", "id_incendi"])
    .reset_index(drop=True)
)
assert merged.id_incendi.is_unique
merged["codi_causa"] = merged["codi_causa"].astype("Int64")
merged.to_csv(folder / "CleanData.csv", index=False, date_format="%Y-%m-%d")
print(f"Saved {len(merged):,} records to {folder / 'CleanData.csv'}")

Records without a comarca code: 14
Saved 16,470 records to Clean data/CleanData.csv
